# Submissão 2 — Modelo A (RoBERTa-base)

In [1]:
import sys
sys.path.insert(0,'../src')

import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from src.transformer_utils import InferenceDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

ID2LABEL_OUT = {
    0: 'Google',
    1: 'Anthropic',
    2: 'Meta',
    3: 'OpenAI',
    4: 'Human'
}

device: cuda


## *carregar dados e modelo*

In [2]:
df_subm = pd.read_csv('subm2.csv', sep=';')

texts = df_subm['Text'].fillna('').tolist()

print(f'amostras a classificar: {len(texts)}')
print(df_subm.head(3))

amostras a classificar: 150
       ID                                               Text
0  D2-101  Microbial mats of coexisting bacteria and arch...
1  D2-102  The origin of life on Earth remains a complex ...
2  D2-103  Estimates of the time at which life arose on E...


In [3]:
MODEL_PATH = '../models/model_roberta'
MAX_LEN    = 128

tokenizer = RobertaTokenizer.from_pretrained(MODEL_PATH)
model     = RobertaForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()
print('modelo RoBERTa carregado!')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

modelo RoBERTa carregado!


In [4]:
subm_ds     = InferenceDataset(texts, tokenizer, MAX_LEN)
subm_loader = DataLoader(subm_ds, batch_size=32)
print(f'batches: {len(subm_loader)}')

batches: 5


## *inferência*

In [5]:
preds_all = []

with torch.no_grad():
    for batch in subm_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = outputs.logits.argmax(dim=1).cpu().tolist()
        preds_all.extend(preds)

df_subm['Label'] = [ID2LABEL_OUT[p] for p in preds_all]
print(df_subm['Label'].value_counts())

Label
Meta         47
Human        43
Google       32
Anthropic    15
OpenAI       13
Name: count, dtype: int64


In [6]:
filename = 'subm2-g2-MEI-A.csv'
df_subm[['ID', 'Label']].to_csv(filename, index=False, sep=';')
print(f"Ficheiro '{filename}' guardado com sucesso!")
print(df_subm[['ID', 'Label']].head())

Ficheiro 'subm2-g2-MEI-A.csv' guardado com sucesso!
       ID   Label
0  D2-101   Human
1  D2-102  OpenAI
2  D2-103   Human
3  D2-104  Google
4  D2-105  Google
